# M3 — EDA: bid book, volumes and no-buy patterns (mFRR)

Learning-bidders track (M3). Sprint 1 cards:
- *EDA: bid book, volumes and no-buy patterns*
- *Prototype descriptive acceptance/rejection reaction analysis*

Start with the **weekly** auctions; check whether the bid statistics change **before a spike**.

Sections
1. Bid-book statistics per auction
2. Weekly spikes: definition
3. Event study: bid book before, at and after the start of a spike
4. Accepted vs rejected bids
5. Can we follow a bidder? (proxy feasibility for Week 2)
6. No-buy auctions
7. Summary and limitations

Everything "before a spike" uses only auctions whose result is published before the spike auction's gate closure (weekly: previous week's result is known 1 week before).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

from src.data.data import build_auctions

OFFERS = ROOT / "data/raw/offers"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

# palette: categorical slots 1-2 + neutral (dataviz default palette)
C1, C2, GREY, INK = "#2a78d6", "#eb6834", "#9a9893", "#52514e"
DIR_COLOR = {"downwards": C1, "upwards": C2}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "lines.linewidth": 2,
    "legend.frameon": False,
})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

## 1. Bid-book statistics per auction

In [ ]:
offers = pd.read_parquet(OFFERS, filters=[("auction_type", "==", "mFRR")])
offers = offers[~offers["is_anticipated"]].copy()
auctions = build_auctions(str(OFFERS), product="mFRR")

mp = offers[offers["accepted"]].groupby("auction_id")["power_price"].max().rename("mp")
offers = offers.join(mp, on="auction_id")
offers["rel"] = np.log10(offers["power_price"] / offers["mp"])   # log10(bid / marginal)


def book_stats(o: pd.DataFrame) -> pd.DataFrame:
    acc, rej = o[o["accepted"]], o[~o["accepted"]]
    near = o[(o["rel"].abs() <= np.log10(1.1))]                       # within ±10 % of marginal
    s = pd.DataFrame({
        "n_acc": acc.groupby("auction_id").size(),
        "n_rej": rej.groupby("auction_id").size(),
        "acc_p10": acc.groupby("auction_id")["power_price"].quantile(0.1),
        "min_rej_above": rej[rej["power_price"] > rej["mp"]].groupby("auction_id")["power_price"].min(),
        "vol_near": near.groupby("auction_id")["offered_volume"].sum(),
    })
    return s


A = auctions.merge(book_stats(offers), left_on="auction_id", right_index=True, how="left")
A[["n_acc", "n_rej", "vol_near"]] = A[["n_acc", "n_rej", "vol_near"]].fillna(0)
A["cover"] = A["offered"] / A["awarded"].replace(0, np.nan)       # offered / awarded
A["rej_share_vol"] = 1 - A["awarded"] / A["offered"]
A["spread_acc"] = np.log10(A["marginal"] / A["acc_p10"])           # spread of accepted prices
A["gap_above"] = np.log10(A["min_rej_above"] / A["marginal"])      # headroom to next rejected bid
A["near_share"] = A["vol_near"] / A["offered"]

# weekly auctions, without the one 2019 correction auction (KORR)
W = A[(A["freq"] == "weekly") & ~A["auction_id"].str.contains("KORR")].sort_values("date")
STATS = ["n_bids", "offered", "awarded", "cover", "rej_share_vol", "n_rej",
         "near_share", "spread_acc", "gap_above"]
W.groupby("direction")[STATS].median().T.round(2)

**What each statistic means** (computed per auction; the table shows the median over all weekly auctions of each direction, i.e. a "typical" weekly auction):

| Statistic | Meaning |
|---|---|
| `n_bids` | number of bids submitted in the auction |
| `offered` | total MW offered by all bids |
| `awarded` | total MW Swissgrid bought |
| `cover` | offered ÷ awarded: MW on offer per MW bought (competition) |
| `rej_share_vol` | share of the offered MW that was rejected |
| `n_rej` | number of rejected bids |
| `near_share` | share of the offered MW priced within ±10 % of the marginal price |
| `spread_acc` | log10(marginal ÷ 10th-percentile accepted price): how far below the marginal price the cheap accepted bids are |
| `gap_above` | log10(cheapest rejected bid above the marginal ÷ marginal): how far above the marginal price the next rejected bid is |

Yearly medians of the weekly bid book. The **cover ratio** (offered / awarded) is the competition measure.

In [ ]:
yearly = W.groupby([W["date"].dt.year, "direction"])[["n_bids", "offered", "awarded", "cover"]].median()
yearly.unstack().round(1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6.4), sharex=True)
for ax, col, title in zip(axes.flat, ["n_bids", "offered", "awarded", "cover"],
                          ["Bids per auction", "Offered volume (MW)", "Awarded volume (MW)",
                           "Cover ratio (offered / awarded)"]):
    for d, g in W.groupby("direction"):
        r = g.set_index("date")[col].rolling(8, min_periods=4).median()
        ax.plot(r.index, r.values, color=DIR_COLOR[d], label=d)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
axes[0, 0].legend()
fig.suptitle("Weekly mFRR bid book, 8-week rolling median", x=0.01, ha="left", fontsize=11)
fig.tight_layout(); save(fig, "m3_weekly_book_over_time")


## 2. Weekly spikes: definition

The weekly price shows **spikes and decays**. A spike is therefore defined by its **shape**, per direction:

1. **Fast rise:** the price reaches a local peak at least **0.2 log10 (×1.58)** above its lowest point of the previous **4 weeks**;
2. **Decay:** within the next **8 weeks** it falls back by at least **30 % of that rise** (it does not have to return to the start: prices often settle higher);
3. **Every sharp peak counts:** peaks only need to be at least **3 weeks** apart, so several sharp spikes in a busy period are each kept (no merging).

Each spike has a **peak** (●, the spike as the eye sees it) and an **onset** (○, the low point where the rise starts).
The analysis "before a spike" is aligned on the **first week of the rise** (onset + 1).

*Not a definition from the brief or the Swissgrid documents: our choice. An earlier rule ("+41 % vs last week") marked slow climbs several times and missed rises built over a few weeks.*


In [ ]:

from scipy.signal import find_peaks

# --- spike definition parameters (judgement calls, not from the brief) ---
RISE = 0.2         # minimum rise from onset to peak, in log10 (0.2 = price x1.58). Higher -> fewer, bigger spikes
UP_WEEKS = 4       # the rise must happen within this many weeks before the peak. Higher -> slower climbs also count
DECAY_WEEKS = 8    # window after the peak in which the price must fall back. Higher -> slower decays also count
DECAY_FRAC = 0.3   # the price must fall back by at least this share of the rise. Higher -> only spikes that fully reverse
MIN_GAP = 3        # minimum weeks between two peaks. Higher -> close peaks in a busy period are dropped


def spike_shape(y: pd.Series) -> tuple[list[int], list[int]]:
    # positions of spike peaks and of their onsets (start of the rise) in the weekly series y.
    # Two close peaks may share the same onset (one rise with two tops): both are kept.
    assert RISE > 0 and 0 <= DECAY_FRAC <= 1, "RISE must be > 0 and DECAY_FRAC between 0 and 1"
    assert UP_WEEKS >= 1 and DECAY_WEEKS >= 1 and MIN_GAP >= 1, "UP_WEEKS, DECAY_WEEKS and MIN_GAP must be >= 1 week"
    v = y.interpolate().to_numpy()
    candidates, _ = find_peaks(v, distance=MIN_GAP)    # local maxima, at least 3 weeks apart
    peaks, onsets = [], []
    for p in candidates:
        lo = max(0, p - UP_WEEKS)
        start = lo + int(np.argmin(v[lo:p]))             # lowest point in the weeks before the peak
        rise = v[p] - v[start]
        if rise < RISE:
            continue                                      # not a fast enough rise
        after = v[p + 1:p + 1 + DECAY_WEEKS]
        if len(after) == 0 or v[p] - after.min() < DECAY_FRAC * rise:
            continue                                      # no decay afterwards
        peaks.append(p)
        onsets.append(start)
    return peaks, onsets


def weekly_panel(direction: str) -> pd.DataFrame:
    g = W[W["direction"] == direction].set_index("date").asfreq("7D")
    g["dy"] = g["y"].diff()
    peaks, onsets = spike_shape(g["y"])
    pos = lambda idx: pd.Series(np.isin(np.arange(len(g)), idx), index=g.index)
    g["peak"] = pos(peaks)
    g["onset"] = pos(onsets)
    g["rise_start"] = pos([min(s + 1, p) for s, p in zip(onsets, peaks)])   # first week of the rise
    episode = np.zeros(len(g), dtype=bool)
    for s, p in zip(onsets, peaks):
        episode[s:p + 2] = True                                             # onset .. peak + 1 week
    g["in_episode"] = episode
    g.attrs["onset_peak"] = list(zip(onsets, peaks))                        # exact (onset, peak) pairs
    g["rise"] = np.nan
    g.loc[g["peak"], "rise"] = [g["y"].iloc[p] - g["y"].iloc[s] for s, p in zip(onsets, peaks)]
    return g


panels = {d: weekly_panel(d) for d in ["downwards", "upwards"]}

Sawtooth: most weeks the price slides down slowly (median weekly change ≈ −6.5 %/week downwards, −4.5 %/week upwards) and spikes are the sudden rises (median ×2.8 down / ×2.5 up, reached in ~3 weeks).

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(13, 5.5), sharex=True)
for ax, (d, g) in zip(axes, panels.items()):
    ax.plot(g.index, g["y"], color=DIR_COLOR[d], linewidth=1.5)
    ax.scatter(g.index[g["peak"]], g.loc[g["peak"], "y"], s=30, color=INK, zorder=3, label="spike (peak)")
    ax.scatter(g.index[g["onset"]], g.loc[g["onset"], "y"], s=28, facecolors="none", edgecolors=INK,
               zorder=3, label="rise starts")
    ax.set_title(f"Weekly {d}: log10 marginal price (CHF/MW per week) — {int(g['peak'].sum())} spikes",
                 loc="left", fontsize=10, color=INK)
axes[0].legend(loc="upper left")
fig.tight_layout(); save(fig, "m3_weekly_spikes")


## 3. The bid book around a spike

All spikes are lined up on **week 0 = the first week of the rise**. Each line is the **median over all spikes** of how far a statistic is from its **usual level** (in %); dotted lines = the same for normal weeks, as a reference.

- *Usual level* = median of the 8 weeks ending at week −3, so slow long-term trends don't count.
- *Grey area* = weeks already published when week 0's auction closes → information a bidder actually has.

In [ ]:

LOG = ["n_bids", "offered", "awarded", "cover", "n_rej"]   # compared in relative terms (log10)


def to_log(g: pd.DataFrame, c: str) -> pd.Series:
    return np.log10(g[c] + 1) if c == "n_rej" else np.log10(g[c])   # n_rej can be 0
LAGS = range(-4, 5)   # k: statistic of week t+k, t = first week of the rise


def event_table(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    for c in LOG:
        g[c] = to_log(g, c)
    normal = ~g["in_episode"] & g["dy"].notna()
    rows = []
    for c in STATS + ["y"]:
        base = g[c].shift(3).rolling(8, min_periods=5).median()
        for k in LAGS:
            dev = g[c].shift(-k) - base
            rows.append({"stat": c, "k": k,
                         "spike": dev[g["rise_start"]].median(), "normal": dev[normal].median()})
    return pd.DataFrame(rows)

ev = pd.concat({d: event_table(g) for d, g in panels.items()}, names=["direction"]).reset_index(level=0)

In [ ]:
show = [("offered", "Offered volume (MW)"), ("n_bids", "Bids"), ("awarded", "Awarded volume (MW)"),
        ("n_rej", "Rejected bids"), ("cover", "Cover ratio (offered / awarded)"), ("y", "Marginal price")]
pct = lambda x: 100 * (10 ** x - 1)          # log10 deviation -> % difference from the usual level
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, (c, title) in zip(axes.flat, show):
    for d in ["downwards", "upwards"]:
        e = ev[(ev["direction"] == d) & (ev["stat"] == c)]
        ax.plot(e["k"], pct(e["spike"]), color=DIR_COLOR[d], marker="o", markersize=4, label=f"{d}: spike")
        ax.plot(e["k"], pct(e["normal"]), color=DIR_COLOR[d], linestyle=":", linewidth=1.5, label=f"{d}: normal")
    ax.axhline(0, color=GREY, linewidth=0.8)
    ax.axvline(0, color=GREY, linewidth=1); ax.axvspan(-4.3, -0.5, color="#f1f0ec", zorder=0)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
for ax in axes[:, 0]:
    ax.set_ylabel("% vs usual level")
for ax in axes[1]:
    ax.set_xlabel("weeks relative to the first week of the rise")
fig.suptitle("Weekly mFRR around price spikes: % difference from the usual level, median over all spikes "
             "(shaded = known before gate closure)", x=0.01, ha="left", fontsize=11)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper center", ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, 0.955), frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, "m3_prespike_event_study")

Before a weekly price spike, fewer bidders show up and less MW is offered, while Swissgrid's need stays the same: less competition → higher price.

**Is the difference real or luck?** For the week just before the rise (information a bidder has), compare all spike weeks with all normal weeks. *p < 0.05* = very unlikely to be chance.

In [ ]:

def t1_deviation(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    for c in LOG:
        g[c] = to_log(g, c)
    D = pd.DataFrame({c: g[c].shift(1) - g[c].shift(3).rolling(8, min_periods=5).median() for c in STATS})
    D["spike"] = g["rise_start"]
    keep = g["rise_start"] | (~g["in_episode"] & g["dy"].notna())
    return D[keep].dropna()

D = pd.concat([t1_deviation(g) for g in panels.values()])
CHECK = {"offered": "Offered volume (MW)", "n_bids": "Bids", "awarded": "Awarded volume (MW) — demand"}
rows = []
for c, label in CHECK.items():
    s, n = D.loc[D["spike"], c], D.loc[~D["spike"], c]
    p = mannwhitneyu(s, n).pvalue                      # Mann-Whitney: are the two groups different?
    rows.append({"week before the rise": label,
                 "spike vs usual": f"{100 * (10 ** s.median() - 1):+.0f} %",
                 "normal vs usual": f"{100 * (10 ** n.median() - 1):+.0f} %",
                 "p-value": "< 0.0001" if p < 1e-4 else f"{p:.4f}",
                 "verdict": "real" if p < 0.05 else "not significant"})
print(f"{int(D['spike'].sum())} spike weeks vs {int((~D['spike']).sum())} normal weeks")
pd.DataFrame(rows).set_index("week before the rise")

## 4. Accepted vs rejected bids

`rel = log10(bid price / marginal price)` of the same auction: 0 = at the marginal price.

In [ ]:
B = offers[offers["mp"].notna()]          # bids of auctions with a marginal price (no-buy auctions excluded)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=False)
bins = np.linspace(-1.5, 2.5, 81)
for ax, f in zip(axes, ["weekly", "daily"]):
    x = B[B["auction_frequency"] == f]
    ax.hist(x.loc[x["accepted"], "rel"].clip(-1.5, 2.5), bins=bins, color=C1, alpha=0.85, label="accepted")
    ax.hist(x.loc[~x["accepted"], "rel"].clip(-1.5, 2.5), bins=bins, color=C2, alpha=0.6, label="rejected")
    ax.axvline(0, color=INK, linewidth=1)
    ax.set_title(f"{f}: bid price relative to marginal, log10", loc="left", fontsize=10, color=INK)
    ax.set_xlabel("log10(bid / marginal)"); ax.set_ylabel("bids")
axes[0].legend()
fig.tight_layout(); save(fig, "m3_accepted_vs_rejected_price")

There are thousands of auctions, each with its own final price, maybe 100, maybe 2,000. Because we measure each bid compared with its own auction's final price, every auction's line lines up at 0.

Drop all the dots from all the auctions onto one line, then count how many dots fall in each small band. That count is the height of each bar.

- Tall blue bar just left of 0: in most auctions, many winning bids are just a bit cheaper than the final price, like the 95 and the 100.
- Tall orange bar just right of 0: in most auctions, many losing bids are just a bit too expensive, like the 105 and the 110. Near-misses.
- Long orange tail to the right: some losers were way too expensive, like the 300.
- Weekly (left panel) has narrow piles: everyone bids close to the final price. Daily (right) has wide piles: bids are all over the place.

**Out-of-merit rejections.** The brief says a bid below the marginal price is *almost always* accepted. Here, in ~46 % of
auctions at least one cheaper bid is rejected. Is it because the bid was larger than the volume still needed (indivisible bid)?

In [ ]:
X = B.sort_values(["auction_id", "power_price"]).copy()
X["awarded_cheaper"] = X.groupby("auction_id")["awarded_volume"].cumsum() - X["awarded_volume"]
X["remaining_need"] = X.groupby("auction_id")["awarded_volume"].transform("sum") - X["awarded_cheaper"]
oom = X[~X["accepted"] & (X["power_price"] < X["mp"])]
both = B.groupby("auction_id")["accepted"].agg(["any", "all"])
n_two_sided = int((both["any"] & ~both["all"]).sum())
n_oom_auctions = oom["auction_id"].nunique()
n_partial = int(((B["awarded_volume"] > 0) & (B["awarded_volume"] < B["offered_volume"])).sum())
share_too_big = (oom["offered_volume"] > oom["remaining_need"]).mean()

print(f"Auctions with both accepted and rejected bids:            {n_two_sided:>7,}")
print(f"  of which reject a bid CHEAPER than the marginal price")
print(f"  (out-of-merit rejection):                                {n_oom_auctions:>7,}  = {n_oom_auctions / n_two_sided:.1%}")
print(f"Out-of-merit rejected bids in total:                       {len(oom):>7,}")
print(f"  explained by size (bid larger than the remaining need):  {share_too_big:.1%}")
print(f"  not explained by size:                                   {1 - share_too_big:.1%}")
print(f"Partially accepted bids (awarded < offered):               {n_partial:>7,}")
for f in ["weekly", "daily"]:
    below = B[(B["auction_frequency"] == f) & (B["power_price"] < B["mp"])]
    print(f"Bids strictly below the marginal price that were accepted ({f}): {below['accepted'].mean():.1%}")

**One example auction** (`TRL_19_KW02_down`, weekly downwards, January 2019): all bids, cheapest first, as they are in the data. Look at the bid with **offered 13 MW, awarded 11 MW** at the marginal price (264.4).

In [ ]:
ex = offers[offers["auction_id"] == "TRL_19_KW02_down"].sort_values("power_price").reset_index(drop=True)
first = ex.iloc[0]
delivery_end = first["date"] + pd.Timedelta(days=6)
print(f"Auction ID       : {first['auction_id']}   (Swissgrid name: {first['auction']})")
print(f"Product          : mFRR, {first['auction_frequency']}, {first['direction']}")
print(f"Delivery period  : {first['date']:%a %d %b %Y} -> {delivery_end:%a %d %b %Y}  (week {first['date'].isocalendar().week} of {first['date'].year})")
print(f"Gate closure     : {first['gate_closure_time_utc'].tz_convert('Europe/Zurich'):%a %d %b %Y, %H:%M} Zurich time")
print(f"Bids             : {len(ex)}  |  offered {ex['offered_volume'].sum()} MW  |  bought {ex['awarded_volume'].sum()} MW"
      f"  |  marginal price {ex.loc[ex['accepted'], 'power_price'].max():.2f} CHF/MW")

ex["result"] = np.select([ex["awarded_volume"] == ex["offered_volume"], ex["awarded_volume"] > 0],
                         ["accepted", "PARTIALLY accepted"], "rejected")
table = ex[["offered_volume", "awarded_volume", "power_price", "result"]].rename(columns={
    "offered_volume": "offered (MW)", "awarded_volume": "awarded (MW)", "power_price": "price (CHF/MW)"})
table.index = range(1, len(table) + 1); table.index.name = "bid (cheapest first)"
display(table.style.format({"price (CHF/MW)": "{:.2f}"})
        .apply(lambda r: ["background-color: #eda100; color: #0b0b0b; font-weight: bold"] * len(r)
               if r["result"] == "PARTIALLY accepted" else [""] * len(r), axis=1))

## 5. Can we follow a bidder? (comparable groups for Week 2)

No bidder IDs. Idea: use the bid **volume** as a fingerprint between consecutive weekly auctions of the same direction.

In [ ]:
print("most common bid sizes (MW):", offers["offered_volume"].value_counts().head(10).to_dict())

tracked, total = 0, 0
for d, g in offers[(offers["auction_frequency"] == "weekly") & ~offers["auction_id"].str.contains("KORR")].groupby("direction"):
    vols = g.groupby("auction_id")["offered_volume"].apply(lambda v: v.value_counts())
    ids = g.groupby("auction_id")["date"].first().sort_values().index
    for prev, cur in zip(ids[:-1], ids[1:]):
        vp, vc = vols.loc[prev], vols.loc[cur]
        tracked += len(set(vp[vp == 1].index) & set(vc[vc == 1].index))
        total += int(vc.sum())
print(f"bids with a volume unique in both consecutive weekly auctions: {tracked} / {total} = {tracked / total:.1%}")

**Answer: no, we cannot track individual bidders by bid size.** Only 8.4 % of weekly bids have a size that identifies them from one auction to the next, because most bidders use the same round sizes (5, 10, 20, 100 MW).

Week 2 options: match on (size, nearest price) between consecutive auctions, or study the **whole bid book** (how the supply curve shifts) instead of individual bidders.

## 6. No-buy auctions (daily only — every weekly auction buys something)

In [ ]:
Dd = A[A["freq"] == "daily"].copy()
Dd["no_buy"] = Dd["y"].isna()
(Dd.groupby([Dd["date"].dt.year, "direction"])["no_buy"].mean().unstack()
   .mul(100).round(0).astype(int).astype(str) + " %")   # share of no-buy auctions per year

*Note:* share of daily auctions where Swissgrid bought **nothing** (every bid rejected), per year and direction. It grows from ~5 % (2019) to ~19 % (2023), jumps to **~90 % in 2024** (daily buying almost suspended), and is **0 % in 2025**.

Same thing **per month**: each cell = % of that month's daily auctions where Swissgrid bought nothing (**0** = always bought something, **100** = bought nothing all month; empty = no data yet).

In [ ]:
# % of the month's daily auctions where Swissgrid bought nothing (100 % = nothing bought all month)
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
for d in ["downwards", "upwards"]:
    t = (Dd[Dd["direction"] == d].groupby([Dd["date"].dt.year, Dd["date"].dt.month])["no_buy"].mean()
         .unstack().reindex(columns=range(1, 13)))
    t.index.name, t.columns = "year", MONTHS
    print(f"\n{d.upper()}: % of daily auctions with nothing bought, per month")
    display((t * 100).map(lambda v: "" if pd.isna(v) else f"{v:.0f}"))   # empty = month not in the data yet

No-buy periods come in **whole delivery weeks** (runs of 42 blocks = 7 days × 6 blocks). Check the week-level structure and the link to the weekly auction of the same week (which closes before the daily ones):

In [ ]:
Dd["week"] = Dd["date"] - pd.to_timedelta(Dd["date"].dt.day_of_week, unit="D")
wk = Dd.groupby(["week", "direction"])["no_buy"].mean().rename("nb_share").reset_index()
aon = ((wk["nb_share"] == 0) | (wk["nb_share"] == 1)).mean()
print(f"Weeks where the daily auctions bought every time OR never (all-or-nothing): {aon:.1%}")

wa = W.groupby(["date", "direction"])["awarded"].sum().rename("weekly_awarded").reset_index().rename(columns={"date": "week"})
wk = wk.merge(wa, on=["week", "direction"])
wk = wk[wk["week"].dt.year != 2024]          # 2024 = near-total suspension, see below
rows = []
for d, g in wk.sort_values("week").groupby("direction"):
    g = g.copy()
    g["weekly_awarded_rel"] = np.log10(g["weekly_awarded"] / g["weekly_awarded"].shift(1).rolling(8, min_periods=4).median())
    g = g.dropna(subset=["weekly_awarded_rel"]); full = g["nb_share"] == 1
    p = mannwhitneyu(g.loc[full, "weekly_awarded_rel"], g.loc[~full, "weekly_awarded_rel"]).pvalue
    pct = lambda x: f"{round(100 * (10 ** x - 1)) + 0:+d} %"
    rows.append({"direction": d,
                 "weeks with NO daily purchase": int(full.sum()),
                 "weekly auction bought vs usual, in those weeks": pct(g.loc[full, "weekly_awarded_rel"].median()),
                 "weekly auction bought vs usual, other weeks": pct(g.loc[~full, "weekly_awarded_rel"].median()),
                 "p-value": "< 0.0001" if p < 1e-4 else f"{p:.4f}",
                 "verdict": "real" if p < 0.05 else "not significant"})
pd.DataFrame(rows).set_index("direction")

### Key finding: daily no-buy weeks are predictable from the weekly auction

- **What we found:** in weeks where the daily auctions bought nothing, the weekly auction of that week had bought **12–18 % more than usual**. Swissgrid had already covered its need in the weekly auction, and the weekly auction closes first, so a bidder can see it coming.
- **Careful:** a big weekly purchase is a **warning sign**, not a guarantee. We have not yet checked how often a big weekly purchase really leads to no daily purchase.

In [ ]:
runs = []
for d, g in Dd.sort_values(["date", "block"]).groupby("direction"):
    r = (g["no_buy"] != g["no_buy"].shift()).cumsum()
    t = g.groupby(r).agg(no_buy=("no_buy", "first"), start=("date", "min"), end=("date", "max"), blocks=("no_buy", "size"))
    t["direction"] = d
    runs.append(t[t["no_buy"] & (t["blocks"] >= 200)])
pd.concat(runs)[["direction", "start", "end", "blocks"]]

at least 200 auctions in a row where no buy happened

## 7. Summary

### Big conclusions
1. **Before a weekly price spike, the bid book thins out:** about **9 % fewer bids** and **8 % less MW offered** than usual, while Swissgrid's demand stays the same (both p < 0.001). Spikes look supply-driven; the signal is real but moderate.
2. **Competition fell while prices rose:** offered MW per MW bought went from 4.1× (2019) to 1.6–1.9× (2021–24) downwards, and from 3.0× to 1.8–2.6× upwards.
3. **Bidders crowd around the final price:** winners just below it, many losers just above it, so a few % decide who wins (weekly tight, daily much wider).
4. **"Below the marginal price = win" is a good approximation for weekly, weaker for daily:** 95 % of weekly bids strictly below the marginal price were accepted, and 82 % of daily ones. Cheaper bids do sometimes lose (at least one in 46 % of auctions); bid size explains only 25 % of those cases.
5. **Individual bidders cannot be tracked by bid size** (only 8 %) → Week 2 needs another way (size + price, or the whole bid book).
6. **Daily no-buy is not random:** it happens in whole weeks, when the weekly auction already bought 12–18 % more than usual; daily buying was almost **suspended Jan–Nov 2024**, none in 2025.

### Current direction
Choices for the first version of the bidding agent:
- **Goal:** maximise the **yearly revenue** of one bidder.
- **Winning rule:** the bid wins if it is **strictly below** the marginal price and is then paid its own price; otherwise it earns 0.
- **Size:** always **1 MW**, so the only decision is the price.
- **Auctions:** **weekly only** to start (upwards and downwards treated separately for now).
- **What the agent sees:** marginal prices already published before its deadline, not individual bids, so partially awarded bids are not shown to it. They stay in the data because they help set the marginal price.
- **Not in this first version:** daily auctions, choosing the volume, simulating the other bidders.

*Limitations:* no bidder IDs; spike definition is our choice; 50 spikes is a small sample; some bids are awarded fewer MW than offered (e.g. 13 → 11 MW), probably because one offer can list several sizes, but there is no offer ID to check it; daily auctions are completely missing on 31 days (downwards) / 32 days (upwards), all between April and November 2024, when daily buying was almost suspended for an unknown reason.